## Phase 1: Browser Initialization & Network Handshake
This phase covers launching your automation instance and establishing a secure connection with the public government server.

| Encountered Error | Real-World Cause | Production-Grade Mitigation Strategy |
| :--- | :--- | :--- |
| `SessionNotCreatedException` | Local Chrome browser auto-updated in the background, breaking driver version compatibility. | Use `webdriver-manager` to auto-fetch matching binaries, or switch to Playwright which auto-manages browser binaries. |
| `WebDriverException` | Local system path conflicts, missing binary execution permissions, or lack of system memory. | Wrap initialization in a `try-except` block; catch the error, log system stats, and gracefully exit. |
| `HTTP 403 Forbidden / CAPTCHA` | The Web Application Firewall (WAF) caught standard Selenium automation headers and blocked your IP. | Deploy `undetected-chromedriver` or use Playwright with stealth plugins to wipe out automation markers. |
| `MaxRetryError / Name Resolution` | Your local machine loses internet connection, or your local DNS fails to resolve the server host. | Implement an exponential backoff retry loop utilizing the `tenacity` library before allowing the main script to fail. |

---

## Phase 2: Server Verification & Target Navigation
This phase covers validating that the government site is actively functional and navigating through categories without breaking execution.

| Encountered Error | Real-World Cause | Production-Grade Mitigation Strategy |
| :--- | :--- | :--- |
| `HTTP 502 / 503 / 504` | Government server is overloaded, crashed, or down for scheduled nightly system maintenance. | Read `driver.title` or check page text for keywords like "Maintenance"; log an alert and exit safely. |
| `HTTP 429 Too Many Requests` | Your bot is navigating too fast, tripping the server's public rate-limiting traffic rules. | Avoid hardcoded loops. Introduce random download delays (jitter) between actions to mimic natural human behavior. |
| `NoSuchElementException` | The specific category link, button, or search field identifier has changed due to an unannounced website layout update. | Use explicit conditional checks (e.g., `if driver.find_elements(...)`) rather than blindly trying to click elements. |
| `TimeoutException` | The search portal takes too long to populate query results due to slow, unoptimized legacy backend databases. | Replace `time.sleep()` with Explicit Waits (`WebDriverWait` with `expected_conditions`) to await elements dynamically. |

---

## Phase 3: Live Data Extraction (Scraping)
This phase covers reading the populated text records off the target pages while keeping the script completely fault-tolerant.

| Encountered Error | Real-World Cause | Production-Grade Mitigation Strategy |
| :--- | :--- | :--- |
| `StaleElementReferenceException` | The site dynamically updates components via AJAX in the background while your script reads the live DOM tree. | Do not parse live elements. Pull the raw page via `driver.page_source`, hand it to Selectolax, and parse it completely offline. |
| `AttributeError / NoneType` | Public data records are structurally inconsistent; a critical field (like an expiration date) is entirely blank on some rows. | Use defensive parsing tricks like `.get_text(strip=True)` inside granular `try-except` wrappers to map missing values to "N/A". |
| `IndexError` | Expected table columns or layout blocks are missing completely because a specific record uses an alternative template layout. | Check sequence boundaries using length validations (`if len(data_cells) >= expected_count:`) before slicing arrays. |

---

## Phase 4: Data Engineering & File Serialization
This phase covers structuring your scraped fields safely and committing them permanently to your disk storage layers.

| Encountered Error | Real-World Cause | Production-Grade Mitigation Strategy |
| :--- | :--- | :--- |
| `PermissionError / File Locked` | Your script tries to write data out to your destination file, but you accidentally left that target CSV open in Microsoft Excel. | Write data records out into uniquely timestamped files, or catch the exception and prompt the user to free the disk lock. |
| `UnicodeEncodeError` | Government data contains specialized localized characters or symbols that cannot map to a standard ASCII file encoder. | Explicitly enforce universal compatibility by initializing file streams with the `encoding="utf-8"` parameter flag. |
| `Total Data Loss on Script Crash` | The script hits a formatting error on record #900 and crashes out, wiping all earlier data held in volatile RAM cache memory. | Open files using Append Mode (`"a"`). Write each processed data row onto the disk storage array immediately inside the loop. |

In [1]:
"""
IMPDS FPS (Food Price Sale) Scraper (web scarpping)
Selenium (Browser automation (Chrome WebDriver))
BeautifulSoup (HTML Prasing and Table extraction)
Pandas ( Data representation)
"""
import os  # Interacts with the operating system 
import re  # REGEX
import json  # JSON Handling
import time  # Manages time related tasks, like pausing the script (sleep) to avoid getting blocked
import argparse  # Allows the script to accept configuration options directly from the command line
from datetime import datetime  # Handles date and time formatting, tracking, and timestamps
from pathlib import Path  # Provides an object-oriented way to work with file and folder paths easily

from selenium import webdriver  # Launches and controls the automated web browser
from selenium.webdriver.common.by import By  # Helps locate elements on a webpage
from selenium.webdriver.support.ui import WebDriverWait, Select  # Handles waiting for elements to load and interacting with dropdown menus
from selenium.webdriver.support import expected_conditions as EC  # Defines conditions to wait for (waiting until a button is clickable)
from selenium.webdriver.chrome.options import Options  # Configures Chrome settings
from bs4 import BeautifulSoup  # Parses HTML source code to easily extract specific text, links, or tables

In [ ]:
def setup_chrome_driver(headless = True):
    """
    Sets up the Chrome WebDriver with specific options for web scraping.
    Returns:
        webdriver.Chrome: Configured Chrome WebDriver instance.
    """
    chrome_options = Options()
    if headless: # Check if headless mode is enabled
        chrome_options.add_argument("--headless") # Run Chrome in headless mode
    chrome_options.add_argument("--disable-gpu")  # Disable GPU acceleration
    chrome_options.add_argument("--no-sandbox")   # Bypass OS security model
    chrome_options.add_argument("--disable-dev-shm-usage")  # Overcome limited resource problems
    chrome_options.add_argument("--window-size=1920,1080")  # Set window size to ensure all elements are visible

    driver = webdriver.Chrome(options = chrome_options) # Initialize the Chrome WebDriver with the specified options
    driver.implicitly_wait(20)  # tells the driver to wait up to 20 sec for elements to appear before giving any eror
    return driver

In [17]:
import time 
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Initialize the driver
driver = webdriver.Chrome()

years = [2026]
months = [3, 4]

for year in years:
    for month in months:
        url = f"https://impds.nic.in/sale/stateUnautmated?month={month}&year={year}#"
        driver.get(url)

        try:
            # target the img specifically using explicit CSS attributes
            goa_element = WebDriverWait(driver, 30).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "img.map[aria-label='GOA']"))
            )
            
            # force click using JavaScript 
            driver.execute_script("arguments[0].click();", goa_element)

            # 3. Wait until transaction state completes 
            WebDriverWait(driver, 15).until(
                EC.text_to_be_present_in_element((By.CSS_SELECTOR, ".status.m_menu"), "Goa")
            )
            
            # 4. Scrape the updated data
            state_data = driver.find_element(By.CSS_SELECTOR, ".counter.convert").text
            print(f"Successfully scraped Goa for {month}/{year}: {state_data}")

        except Exception as e:
            print(f"Error processing {month}/{year}: {e}")

driver.quit()

Error processing 3/2026: Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff75bd98945+14ce5]
	chromedriver!GetHandleVerifier [0x7ff75bd989a0+14d40]
	chromedriver!(No symbol) [0x7ff75b8c5afd]
	chromedriver!(No symbol) [0x7ff75b920b59]
	chromedriver!(No symbol) [0x7ff75b920e5c]
	chromedriver!(No symbol) [0x7ff75b9718d7]
	chromedriver!(No symbol) [0x7ff75b96e4ab]
	chromedriver!(No symbol) [0x7ff75b91308c]
	chromedriver!(No symbol) [0x7ff75b913fb3]
	chromedriver!GetHandleVerifier [0x7ff75c3ce60b+64a9ab]
	chromedriver!GetHandleVerifier [0x7ff75c3c8b12+644eb2]
	chromedriver!GetHandleVerifier [0x7ff75c3ee3ae+66a74e]
	chromedriver!GetHandleVerifier [0x7ff75bdb5d7e+3211e]
	chromedriver!GetHandleVerifier [0x7ff75bdbe52c+3a8cc]
	chromedriver!GetHandleVerifier [0x7ff75bda2854+1ebf4]
	chromedriver!GetHandleVerifier [0x7ff75bda29e4+1ed84]
	chromedriver!GetHandleVerifier [0x7ff75bd85d17+20b7]
	KERNEL32!BaseThreadInitThunk [0x7ff830b47374+14]
	ntdll!RtlUserThreadStart [0x7ff83167cc91+21]

Err